# Homework: AI Orchestration with Kestra

## Question 1: Context Engineering

`prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"`

### ChatGPT Result

Below is an example **Kestra** flow that:

1. Downloads an NYC Taxi CSV file
2. Uploads it to Google Cloud Storage (optional but recommended)
3. Loads the CSV into BigQuery
4. Creates the destination table automatically

Replace the project, dataset, bucket, and service account values with your own.

```yaml
id: nyc_taxi_csv_to_bigquery
namespace: company.team

variables:
  project_id: your-gcp-project
  dataset: nyc_taxi
  table: yellow_tripdata
  bucket: your-gcs-bucket

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2023-01.csv.gz

  - id: decompress
    type: io.kestra.plugin.compress.ArchiveDecompress
    from: "{{ outputs.download_csv.uri }}"

  - id: upload_to_gcs
    type: io.kestra.plugin.gcp.gcs.Upload
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    from: "{{ outputs.decompress.files[0] }}"
    bucket: "{{ vars.bucket }}"
    name: yellow_tripdata_2023-01.csv

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.LoadFromGcs
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT') }}"
    projectId: "{{ vars.project_id }}"
    dataset: "{{ vars.dataset }}"
    table: "{{ vars.table }}"
    from:
      - "gs://{{ vars.bucket }}/yellow_tripdata_2023-01.csv"
    format: CSV
    autodetect: true
    skipLeadingRows: 1
    writeDisposition: WRITE_TRUNCATE
    createDisposition: CREATE_IF_NEEDED
```

### Required setup

* Create a GCS bucket.
* Create a BigQuery dataset.
* Store your GCP service account JSON as a Kestra secret:

  ```
  GCP_SERVICE_ACCOUNT
  ```
* Grant the service account the following IAM roles:

  * BigQuery Data Editor
  * BigQuery Job User
  * Storage Object Admin (or Storage Object Creator)

### Alternative (without GCS)

If your version of the GCP plugin supports loading directly from a local file, you can skip the upload step and use a `Load` task pointing to the decompressed CSV. However, staging the file in GCS is the recommended approach because it is more scalable and aligns with BigQuery's native loading mechanism.




### AI Copilot

# Don't forget to replace 'your_project.your_dataset.nyc_taxi_data' with your actual BigQuery table.
# Ensure your Kestra instance has the necessary GCP credentials configured to access BigQuery.
id: nyc_taxi_data_load
namespace: dev.taxi

tasks:
  - id: download_taxi_data
    type: io.kestra.plugin.core.http.Download
    uri: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.csv
    saveAs: yellow_tripdata.csv

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_taxi_data.uri }}"
    destinationTable: your_project.your_dataset.nyc_taxi_data
    format: CSV
    autodetect: true
    csvOptions:
      skipLeadingRows: 1

The primary reason AI Copilot generates better Kestra flows?
1. AI Copilot has access to current Kestra plugin documentation

## Question 2: RAG vs No RAG




INFO 2026-07-20T09:55:41.460409Z ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features that significantly enhance its capabilities and user experience. Here are 5 major features with brief descriptions:

1.  **Event-Driven Flows (Triggers for External Events):**
    *   **Description:** This was a groundbreaking feature, allowing Kestra workflows to be triggered by external events from various sources. Instead of relying solely on scheduled executions or manual triggers, Kestra 1.1 enabled real-time, event-driven orchestration. This included triggers for AWS SQS, Confluent Kafka, Google Pub/Sub, and more, making Kestra much more responsive and integrated into modern data architectures.

2.  **API Key Authentication for UI and API Access:**
    *   **Description:** Kestra 1.1 introduced a more robust and secure way to manage authentication for both its UI and API. Users could now generate and manage API keys, providing a more granular and secure access control mechanism compared to previous methods. This was crucial for integrating Kestra into automated scripts, CI/CD pipelines, and other external systems while maintaining security best practices.

3.  **Namespace-Level RBAC (Role-Based Access Control):**
    *   **Description:** A significant improvement in security and collaboration, Namespace-Level RBAC allowed administrators to define user roles and permissions at the namespace level. This meant different teams or projects could have their own dedicated namespaces with specific access rights, preventing unauthorized access and ensuring proper segregation of duties within a single Kestra instance.

4.  **Flow-Level Task Defaults:**
    *   **Description:** This feature simplified flow development and maintenance by allowing users to define default values for task properties at the flow level. Instead of repeating the same configuration for multiple tasks within a flow (e.g., a default timeout or an error handling strategy), these defaults could be set once for the entire flow. Individual tasks could still override these defaults if needed, offering flexibility while reducing boilerplate code.

5.  **New Task: `io.kestra.core.tasks.flows.EachSequential`:**
    *   **Description:** Kestra 1.1 introduced the `EachSequential` task, providing a new way to iterate over a list of items and execute a sub-flow for each item *sequentially*. While Kestra already had parallel iteration capabilities, `EachSequential` was vital for scenarios where the order of processing matters, or when a subsequent item's processing depends on the successful completion of the previous one (e.g., when interacting with external APIs that have rate limits or order dependencies).

These features collectively made Kestra 1.1 a much more powerful, secure, and versatile orchestration platform, particularly for event-driven architectures and enterprise deployments requiring fine-grained access control.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.


Kestra 1.1 introduced several major features. Here are at least 5 with brief descriptions:
1. **New Filters**: The UI filters were completely redesigned to be more intuitive and powerful. Users can choose from explicit options, reset filters with a single click, save frequently used combinations, and customize table columns.
2. **No-Code Dashboard Editor**: This feature allows users to create and edit custom dashboards directly from the UI using a no-code, multi-panel editor. It enables building monitoring dashboards, business intelligence views, or executive reports without writing YAML.
3. **Human Task**: (Enterprise Edition) This new task allows workflows to pause and wait for manual approval from specific users or groups before continuing. It's designed for workflows requiring human validation for critical operations.
4. **Multi-Agent AI Systems**: AI agents can now use other AI agents as tools, enabling sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
5. **Fix with AI**: When tasks fail, Kestra 1.1 provides AI-powered suggestions to help diagnose and resolve issues, speeding up troubleshooting and getting executions back on track.
6. **New MailReceivedTrigger and RealTimeTrigger**: These triggers enable workflows to be initiated based on incoming emails and real-time events, supporting both batch processing and one-time event handling for email-driven automations.
7. **Dozens of New Plugins**: Kestra 1.1 significantly expanded its integration capabilities with new plugins for various categories including Data & Database (Liquibase, dlt), SaaS & API (Airtable, Stripe, Shopify, Odoo), Cloud & Infrastructure (Dataform, AWS CloudWatch, Apache Flink), and many more.
🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!

The non-RAG response about Kestra 1.1 features is best described as:
- Vague, generic, or fabricated — the model guesses from training data

## Question 3: Token usage — short summary

What is the approximate output token count for multilingual_agent?
- 79 (60-100 tokens)

## Question 4: Token usage — long summary

Run 4_simple_agent.yaml again with summary_length = long.

Compare the multilingual_agent output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?
- 175 (2-5x more)

## Question 5: Modifying a flow

Open 4_simple_agent.yaml in the Kestra flow editor. Find the english_brevity task and change its prompt from asking for exactly 1 sentence to asking for exactly 3 sentences.

Save the flow, then run it with summary_length = long.

Compare the english_brevity output token count to the original 1-sentence version (also with summary_length = long). How do they compare?

- original 1-sentence version => 49
- 3-sentence version => 92

closest ==>  ~1.9 =>  2-4x more

## Question 6: Best Practices

Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?


- Use traditional task-based workflows for predictability and auditability